# Compare previous protocols
- To quantify baseline fitting issue.
- 1 minute baseline window vs 30 minute baseline window
- 1 minute baseline window is the current dff.

In [ ]:
import sys
from pathlib import Path

repo_code_dir = Path('/single-cell-zdrift-qc/code')
if str(repo_code_dir) not in sys.path:
    sys.path.append(str(repo_code_dir))

import register_fov_local_zstack as reg_mod
import register_fov_local_zstack_methods as methods_mod
import register_fov_local_zstack_parallel as par_mod
import register_fov_local_zstack_qc as qc_mod
import register_fov_local_zstack_viz as viz_mod

ModuleNotFoundError: No module named 'register_fov_local_zstack'

In [15]:
from pathlib import Path
import numpy as np

from lamf_analysis.code_ocean import capsule_data_utils as cdu
from lamf_analysis.code_ocean import docdb_utils
from aind_ophys_utils import dff as dff_utils

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [9]:
data_dir = Path('/root/capsule/data')
subject_ids = [755252, 767018, 767022, 783551, 785054, 782149, 788406, 790322, 800792, 800995, 804363, 804670] # all Slc32a1;Oi1 collected so far


In [26]:
docdb_utils.get_derived_data_assets(subject_ids[-1], 'single-cell-zdrift',
                                        filter_based_on_date_format=True).dropna()

,derived_name,location,derived_asset_id,process,derived_date,raw_name
0,multiplane-ophys_804670_2025-09-12_09-36-10_si...,s3://codeocean-s3datasetsbucket-1u41qdg42ur9/3...,353e97e8-4e81-4986-bae8-d2215231cac5,"{'name': 'Other', 'software_version': '0.1.0',...",2026-04-03,multiplane-ophys_804670_2025-09-12_09-36-10_si...
1,multiplane-ophys_804670_2025-09-20_09-16-09_si...,s3://codeocean-s3datasetsbucket-1u41qdg42ur9/d...,d909e23b-db4a-40b6-944b-995fd0fad318,"{'name': 'Other', 'software_version': '0.1.0',...",2026-04-03,multiplane-ophys_804670_2025-09-20_09-16-09_si...
2,multiplane-ophys_804670_2025-09-30_10-05-45_si...,s3://codeocean-s3datasetsbucket-1u41qdg42ur9/d...,daace940-4c4b-4d77-86bc-3a50671b98ab,"{'name': 'Other', 'software_version': '0.1.0',...",2026-04-03,multiplane-ophys_804670_2025-09-30_10-05-45_si...
3,multiplane-ophys_804670_2025-09-25_09-49-47_si...,s3://codeocean-s3datasetsbucket-1u41qdg42ur9/c...,c8f5b0a8-49fb-4aaa-95ce-26057b428f6f,"{'name': 'Other', 'software_version': '0.1.0',...",2026-04-03,multiplane-ophys_804670_2025-09-25_09-49-47_si...
4,multiplane-ophys_804670_2025-09-11_09-40-54_si...,s3://codeocean-s3datasetsbucket-1u41qdg42ur9/4...,42a002b3-1e5a-4f75-a8d9-f866c03ec288,"{'name': 'Other', 'software_version': '0.1.0',...",2026-04-03,multiplane-ophys_804670_2025-09-11_09-40-54_si...
5,multiplane-ophys_804670_2025-10-02_10-10-21_si...,s3://codeocean-s3datasetsbucket-1u41qdg42ur9/e...,ec4074b4-ebfd-4acd-b0f6-648896234666,"{'name': 'Other', 'software_version': '0.1.0',...",2026-04-03,multiplane-ophys_804670_2025-10-02_10-10-21_si...
6,multiplane-ophys_804670_2025-09-24_09-30-56_si...,s3://codeocean-s3datasetsbucket-1u41qdg42ur9/d...,d7656f6c-7bbc-4551-824d-bb79fe15ddd0,"{'name': 'Other', 'software_version': '0.1.0',...",2026-04-03,multiplane-ophys_804670_2025-09-24_09-30-56_si...
7,multiplane-ophys_804670_2025-10-01_10-15-38_si...,s3://codeocean-s3datasetsbucket-1u41qdg42ur9/3...,3c9074e9-3a19-4a21-8089-3ae057223819,"{'name': 'Other', 'software_version': '0.1.0',...",2026-04-03,multiplane-ophys_804670_2025-10-01_10-15-38_si...
8,multiplane-ophys_804670_2025-09-19_09-38-54_si...,s3://codeocean-s3datasetsbucket-1u41qdg42ur9/8...,83e2d118-2fbc-4c74-9063-16284911afe5,"{'name': 'Other', 'software_version': '0.1.0',...",2026-04-03,multiplane-ophys_804670_2025-09-19_09-38-54_si...
9,multiplane-ophys_804670_2025-09-27_09-39-25_si...,s3://codeocean-s3datasetsbucket-1u41qdg42ur9/c...,c648d976-9824-49bc-b73d-f5fadd5e70c1,"{'name': 'Other', 'software_version': '0.1.0',...",2026-04-03,multiplane-ophys_804670_2025-09-27_09-39-25_si...


In [28]:
def get_attach_df(subject_id):
    # including single-cell-zdrift
    session_infos = docdb_utils.get_session_infos_from_docdb(subject_id, filter_test_data=True)
    processed_infos = docdb_utils.get_processed_data_info(subject_id).sort_values('long_window').drop_duplicates(subset=['raw_name'])
    merged_df = processed_infos.merge(session_infos, left_on='raw_name', right_on='raw_asset_name', how='left')

    sczdrift_df = docdb_utils.get_derived_data_assets(subject_id, 'single-cell-zdrift-qc')
    sczdrift_df.rename(columns={'derived_name': 'single_cell_zdrift_derived_name',
                                'derived_asset_id': 'single_cell_zdrift_derived_asset_id'}, inplace=True)
    merged_df = merged_df.merge(sczdrift_df[['raw_name',
                                            'single_cell_zdrift_derived_name',
                                            'single_cell_zdrift_derived_asset_id']],
                                on='raw_name', how='inner')
    # attach_df = merged_df[merged_df.session_type.str.contains('OPHYS_')]
    return merged_df

In [29]:
subject_id = subject_ids[0]
merged_df = get_attach_df(subject_id).sort_values('acquisition_date').reset_index(drop=True)
merged_df.head()

,raw_name,long_window,processed_asset_id,processed_date,processed_name,acquisition_date,session_type,rig_id,session_key,project_name,raw_asset_name,raw_asset_id,s3_path,session_type_exposures,single_cell_zdrift_derived_name,single_cell_zdrift_derived_asset_id
0,multiplane-ophys_755252_2024-11-12_09-43-51,60,6747f968-d6dc-4db6-b6a6-941aac40425e,2025-09-05,multiplane-ophys_755252_2024-11-12_09-43-51_pr...,2024-11-12,TRAINING_0_gratings_autorewards_15min,MESO.1,755252_2024-11-12,Learning mFISH-V1omFISH,multiplane-ophys_755252_2024-11-12_09-43-51,73d47ce2-1d56-48db-9f7b-d99aaf782d1d,s3://aind-private-data-prod-o5171v/multiplane-...,1,multiplane-ophys_755252_2024-11-12_09-43-51_si...,b61cd61a-2aa1-47fb-8619-dc64e15326e7
1,multiplane-ophys_755252_2024-11-13_09-16-29,60,41d3bfb3-f969-4442-936d-58aaf02720d6,2025-09-05,multiplane-ophys_755252_2024-11-13_09-16-29_pr...,2024-11-13,TRAINING_1_gratings,MESO.1,755252_2024-11-13,Learning mFISH-V1omFISH,multiplane-ophys_755252_2024-11-13_09-16-29,207fb83f-3fdf-4879-8f45-f02119557d4e,s3://aind-private-data-prod-o5171v/multiplane-...,1,multiplane-ophys_755252_2024-11-13_09-16-29_si...,4af41f88-0f0a-4909-91d4-4b099724ff11
2,multiplane-ophys_755252_2024-11-14_11-28-00,60,15f5d46e-80d5-422f-8608-461484b4c86b,2025-09-05,multiplane-ophys_755252_2024-11-14_11-28-00_pr...,2024-11-14,TRAINING_1_gratings,MESO.1,755252_2024-11-14,Learning mFISH-V1omFISH,multiplane-ophys_755252_2024-11-14_11-28-00,1b187fe7-13ac-4800-9dba-b6523f22765c,s3://aind-private-data-prod-o5171v/multiplane-...,2,multiplane-ophys_755252_2024-11-14_11-28-00_si...,8c939675-da89-4505-8288-2045c79eff8a
3,multiplane-ophys_755252_2024-11-15_10-49-40,60,f0c72bdf-844e-4d27-bb71-c6d9e879788c,2025-09-05,multiplane-ophys_755252_2024-11-15_10-49-40_pr...,2024-11-15,TRAINING_1_gratings,MESO.1,755252_2024-11-15,Learning mFISH-V1omFISH,multiplane-ophys_755252_2024-11-15_10-49-40,95ba6dcc-c32a-4812-a234-27ae729cc497,s3://aind-private-data-prod-o5171v/multiplane-...,3,multiplane-ophys_755252_2024-11-15_10-49-40_si...,97ea2b7a-deed-492d-9c3d-9eb9d858a15d
4,multiplane-ophys_755252_2024-11-18_08-01-08,60,bccb963f-a624-4ce9-8f67-7faafeb0846e,2025-09-05,multiplane-ophys_755252_2024-11-18_08-01-08_pr...,2024-11-18,TRAINING_1_gratings,MESO.1,755252_2024-11-18,Learning mFISH-V1omFISH,multiplane-ophys_755252_2024-11-18_08-01-08,5f5ffe30-e764-41db-883b-c00c04219413,s3://aind-private-data-prod-o5171v/multiplane-...,4,multiplane-ophys_755252_2024-11-18_08-01-08_si...,4c304ef7-5ed3-4f50-8cb4-b021ed89fb74


In [11]:
session_i = 10
session_row = merged_df.iloc[session_i]
processed_name = session_row.processed_name
processed_dir = data_dir / processed_name
plane_ids = cdu.get_plane_ids_from_processed_path(processed_dir)
intended_depths = [cdu.get_intended_depth(processed_dir / plane_id) for plane_id in plane_ids]

No intended depth found in platform info for /root/capsule/data/multiplane-ophys_755252_2024-12-05_11-34-40_processed_2025-09-05_11-46-59/VISp_0, returning targeted depth as fallback
No intended depth found in platform info for /root/capsule/data/multiplane-ophys_755252_2024-12-05_11-34-40_processed_2025-09-05_11-46-59/VISp_1, returning targeted depth as fallback
No intended depth found in platform info for /root/capsule/data/multiplane-ophys_755252_2024-12-05_11-34-40_processed_2025-09-05_11-46-59/VISp_2, returning targeted depth as fallback
No intended depth found in platform info for /root/capsule/data/multiplane-ophys_755252_2024-12-05_11-34-40_processed_2025-09-05_11-46-59/VISp_3, returning targeted depth as fallback
No intended depth found in platform info for /root/capsule/data/multiplane-ophys_755252_2024-12-05_11-34-40_processed_2025-09-05_11-46-59/VISp_4, returning targeted depth as fallback
No intended depth found in platform info for /root/capsule/data/multiplane-ophys_7552

In [21]:
roi_table = cdu.get_roi_table_from_plane_path(plane_path)

In [22]:
roi_table.head()

,cell_roi_id,mask_matrix,height,width,x,y,centroid,bounding_box,valid_roi,exclusion_labels,pixel_size_um,touching_motion_border,small_roi
0,0,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...",20,18,25,0,"(34.0, 10.0)","[(0, 25, 20, 43)]",False,None,0.78,True,False
1,1,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...",16,17,362,0,"(370.5, 8.0)","[(0, 362, 16, 379)]",False,None,0.78,True,False
2,2,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...",17,18,271,1,"(280.0, 9.5)","[(1, 271, 18, 289)]",False,None,0.78,True,False
3,3,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...",15,15,91,11,"(98.5, 18.5)","[(11, 91, 26, 106)]",True,None,0.78,False,False
4,4,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...",13,16,464,16,"(472.0, 22.5)","[(16, 464, 29, 480)]",True,None,0.78,False,False


In [24]:
F_all = []
dff_short_window_all = []
dff_long_window_all = []
baseline_short_window_all = []
baseline_long_window_all = []
plane_ids_all = []
plane_depths_all = []
valid_roi_inds_all = []
for plane_id in plane_ids:
    plane_path = processed_dir / plane_id
    plane_depth = cdu.get_intended_depth(plane_path)
    roi_table = cdu.get_roi_table_from_plane_path(plane_path)
    valid_roi_inds = roi_table.query('valid_roi').cell_roi_id.values
    valid_roi_inds_all.append(valid_roi_inds)
    F = cdu.load_corrected_fluorescence(plane_path=plane_path)
    F_valid = F[valid_roi_inds, :]
    frame_rate = cdu.get_frame_rate_from_plane_path(plane_path)
    dff_short = cdu.load_dff_from_plane_path(plane_path)
    baseline_short = cdu.get_baseline_traces(plane_path)
    dff_long, baseline_long, _ = dff_utils.dff(F_valid, long_window=60*30, fs=frame_rate)
    F_all.append(F_valid)
    dff_short_window_all.append(dff_short[valid_roi_inds, :])
    dff_long_window_all.append(dff_long)
    baseline_short_window_all.append(baseline_short[valid_roi_inds, :])
    baseline_long_window_all.append(baseline_long)
    plane_ids_all.append([plane_id]*F_valid.shape[0])
    plane_depths_all.append([plane_depth]*F_valid.shape[0])


Using provided plane_path to load corrected fluorescence
Using provided plane_path to load corrected fluorescence
Using provided plane_path to load corrected fluorescence
Using provided plane_path to load corrected fluorescence
Using provided plane_path to load corrected fluorescence
Using provided plane_path to load corrected fluorescence
Using provided plane_path to load corrected fluorescence
Using provided plane_path to load corrected fluorescence


In [17]:
F_all_array = np.concatenate(F_all, axis=0)
dff_short_window_all_array = np.concatenate(dff_short_window_all, axis=0)
dff_long_window_all_array = np.concatenate(dff_long_window_all, axis=0)
baseline_short_window_all_array = np.concatenate(baseline_short_window_all, axis=0)
baseline_long_window_all_array = np.concatenate(baseline_long_window_all, axis=0)

(23557644,)